**DATASTORM 6.0 STORMING ROUND**


We found out that there were 905 unique agents in the training data set and that out of them only 650 had 20 months of data each. Some agents had only 4 months of data.

Also the data was skewed as there were only 2339 entries in the new_policy_count for the next month which were 0 in the training data set. (There were 15309 entries in the dataset)

In [ ]:
#Used this to try and predict the features which affected the prediction the most


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# ─────────────────────────────────────────────
# 1) Load data & create binary target
# ─────────────────────────────────────────────
date_cols = ["year_month", "agent_join_month", "first_policy_sold_month"]

df = pd.read_csv(
    "train_with_weekly_values.csv",
    parse_dates=date_cols
)

df['latest_policy_count'] = (df['latest_policy_count'] > 0).astype(int)

# ─────────────────────────────────────────────
# 2) Split into X / y
# ─────────────────────────────────────────────
drop_cols = ['latest_policy_count']
X = df.drop(columns=drop_cols)
y = df['latest_policy_count']

# ─────────────────────────────────────────────
# 3) Identify column groups
# ─────────────────────────────────────────────
cat_cols = ["agent_code"]
num_cols = [c for c in X.columns if c not in cat_cols + date_cols]

# ─────────────────────────────────────────────
# 4) Train/test split
# ────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ─────────────────────────────────────────────
# 5) Build date-feature extractor with feature names
# ─────────────────────────────────────────────
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df_dates = pd.DataFrame(X, columns=date_cols)
        feats = []
        for col in date_cols:
            df_dates[col] = df_dates[col].fillna(pd.Timestamp('2000-01-01'))
            feats.append(df_dates[col].dt.year.values.reshape(-1, 1))
            feats.append(df_dates[col].dt.month.values.reshape(-1, 1))
        return np.hstack(feats)

    def get_feature_names_out(self, input_features=None):
        # Generate feature names for year and month for each date column
        feature_names = []
        for col in date_cols:
            feature_names.append(f"{col}_year")
            feature_names.append(f"{col}_month")
        return np.array(feature_names)

date_transformer = Pipeline([
    ("extract", DateFeatureExtractor()),
    ("scale", StandardScaler())
])

# ─────────────────────────────────────────────
# 6) Preprocessing pipelines
# ─────────────────────────────────────────────
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer([
    ("num",   numeric_transformer,    num_cols),
    ("cat",   categorical_transformer,cat_cols),
    ("date",  date_transformer,       date_cols)
], remainder="drop")

# ─────────────────────────────────────────────
# 7) Define base classifiers & stacking ensemble
# ─────────────────────────────────────────────
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale = n_neg / n_pos

print(f"Negative: {n_neg}, Positive: {n_pos}, scale_pos_weight = {scale:.2f}")

xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=scale,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

lgb_model = lgb.LGBMClassifier(
    objective="binary",
    class_weight={0:1, 1:1/scale},
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42
)

cat_model = CatBoostClassifier(
    loss_function="Logloss",
    auto_class_weights='Balanced',
    depth=6,
    learning_rate=0.05,
    iterations=400,
    verbose=0,
    random_seed=42
)

ensemble = StackingClassifier(
    estimators=[
        ("xgb", xgb_model),
        ("lgb", lgb_model),
        ("cat", cat_model)
    ],
    final_estimator=LogisticRegressionCV(),
    passthrough=True,
    n_jobs=1
)

# ─────────────────────────────────────────────
# 8) SMOTETomek pipeline & ratio sweep
# ─────────────────────────────────────────────
pipe = ImbPipeline([
    ("preproc",  preprocessor),
    ('smote', SVMSMOTE(random_state=42)),
    ("stack",    ensemble)
])

pipe.fit(X_train, y_train)
y_proba = pipe.predict_proba(X_test)[:,1]
y_preds = pipe.predict(X_test)

print(classification_report(y_test, y_preds, digits=3))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

# ─────────────────────────────────────────────
# 9) Load test data and make predictions
# ─────────────────────────────────────────────
test_df = pd.read_csv(
    "test_with_weekly_values.csv",
    parse_dates=date_cols
)

# Prepare test features (drop target if present, though it might not be)
X_test_final = test_df.drop(columns=['latest_policy_count'] if 'latest_policy_count' in test_df.columns else [])

# Predict on test dataset
y_pred_test = pipe.predict(X_test_final)

# Create output DataFrame
output_df = pd.DataFrame({
    'row_id': test_df['row_id'],
    'target_column': y_pred_test
})

# Save to CSV
output_df.to_csv('predictions.csv', index=False)

print("Predictions saved to 'predictions.csv'")

# ─────────────────────────────────────────────
# 10) Feature Names for SHAP Analysis (Optional)
# ─────────────────────────────────────────────
# Get feature names after preprocessing
feature_names = pipe.named_steps['preproc'].get_feature_names_out()
print("Feature names after preprocessing:", feature_names)

# If you want to proceed with SHAP analysis (requires 'shap' package)
import shap

# Transform the test data through the preprocessor
X_test_transformed = pipe.named_steps['preproc'].transform(X_test_final)

# Use SHAP with the stacking classifier
explainer = shap.KernelExplainer(
    lambda x: pipe.named_steps['stack'].predict_proba(x)[:, 1],
    X_test_transformed
)
shap_values = explainer.shap_values(X_test_transformed)

# Plot SHAP summary (requires matplotlib)
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, plot_type="bar")

List of top factors affecting yearly performance.

1. new_policy_count
2. new_policy_count_lag1
3. unique_proposals_week_1
4. unique_quotations_week_1
5. unique_customers_week_1
6. ANBP_value
7. net_income
8. number_of_policy_holders
9. unique_proposals_week_2
10. new_policy_count_lag2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

# Load the dataset
df = pd.read_csv("train_storming_round.csv", parse_dates=['agent_join_month', 'first_policy_sold_month', 'year_month'])

# Convert categorical and datetime columns to numeric
df['agent_code'] = df['agent_code'].astype('category').cat.codes

for col in ['agent_join_month', 'first_policy_sold_month', 'year_month']:
    df[col] = (df[col] - df[col].min()).dt.days

# Select only numeric columns
num_cols = df.select_dtypes(include='number').columns.tolist()

# Generate all unique pairs of numeric columns
col_pairs = list(combinations(num_cols, 2))

# Plot settings
plots_per_fig = 16
total_pairs = len(col_pairs)
num_figs = (total_pairs + plots_per_fig - 1) // plots_per_fig

# Create plots
for fig_idx in range(num_figs):
    fig, axes = plt.subplots(4, 4, figsize=(20, 15))
    fig.suptitle(f"Scatter Plots (Page {fig_idx + 1})", fontsize=18)
    axes = axes.flatten()

    for i in range(plots_per_fig):
        pair_idx = fig_idx * plots_per_fig + i
        if pair_idx >= total_pairs:
            axes[i].axis('off')
            continue

        x_col, y_col = col_pairs[pair_idx]
        sns.scatterplot(data=df, x=x_col, y=y_col, ax=axes[i], alpha=0.5, s=10)
        axes[i].set_title(f"{x_col} vs {y_col}", fontsize=10)
        axes[i].tick_params(labelsize=8)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Leave space for the title
    plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'train_storming_round.csv'

Plotting the Correlation heatmap

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data into a pandas DataFrame
data = pd.read_csv('train_storming_round.csv')  # Replace with the actual path to your data file

# Display the first few rows of the dataset
print("First few rows of the dataset:")
print(data.head())

# Basic statistics of the dataset
print("\nBasic statistics of the dataset:")
print(data.describe())

# Check for missing values
print("\nMissing values in the dataset:")
print(data.isnull().sum())



# Visualize the distribution of Avg_Temperature
plt.figure(figsize=(10, 6))
sns.histplot(data['ANBP_value'], kde=True, color='blue')
plt.title('Distribution of ANBP_value')
plt.xlabel('ANBP_Value')
plt.ylabel('Frequency')
plt.show()



# Correlation heatmap
plt.figure(figsize=(24, 16))

# Select only numeric columns for correlation
numeric_data = data.select_dtypes(include=['float64', 'int64'])
correlation_matrix = numeric_data.corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()



FileNotFoundError: [Errno 2] No such file or directory: 'train_storming_round.csv'

Added the new_policy_counts of 3 previous months as lag features

In [ ]:
import pandas as pd
import io



columns = [
    "row_id", "agent_code", "agent_age", "agent_join_month",
    "first_policy_sold_month", "year_month",
    "unique_proposals_last_7_days", "unique_proposals_last_15_days",
    "unique_proposals_last_21_days", "unique_proposal",
    "unique_quotations_last_7_days", "unique_quotations_last_15_days",
    "unique_quotations_last_21_days", "unique_quotations",
    "unique_customers_last_7_days", "unique_customers_last_15_days",
    "unique_customers_last_21_days", "unique_customers",
    "new_policy_count", "ANBP_value", "net_income",
    "number_of_policy_holders", "number_of_cash_payment_policies"
]

# Load and parse dates
df = pd.read_csv('/Users/rus1ru/Downloads/data-storm-6-0/train_storming_round.csv', names=columns, header=0)
df['year_month'] = pd.to_datetime(df['year_month'], format='%m/%d/%Y')
df['agent_join_month'] = pd.to_datetime(df['agent_join_month'], format='%m/%d/%Y')

# Sort
df = df.sort_values(['agent_code', 'agent_join_month', 'year_month'])

# Set index
df = df.set_index('year_month')

# Define full monthly index
all_months = pd.date_range(df.index.min(), df.index.max(), freq='MS')

# Reindex per agent manually
processed = []
for code, grp in df.groupby('agent_code'):
    grp_monthly = grp.reindex(all_months)
    grp_monthly['agent_code'] = code
    processed.append(grp_monthly)

df_monthly = pd.concat(processed)
df_monthly.index.name = 'year_month'
df_monthly = df_monthly.reset_index()

# Fill missing values
df_monthly.update(df_monthly.groupby('agent_code').ffill())
df_monthly = df_monthly.fillna(0)

# Create lag features
for lag in range(1, 4):
    df_monthly[f'new_policy_count_lag{lag}'] = (
        df_monthly.groupby('agent_code')['new_policy_count']
                   .shift(lag).fillna(0)
    )

df_monthly.to_csv('/Users/rus1ru/Downloads/data-storm-6-0/output_file.csv', index=False)
print("Processing complete. Output saved to output_file.csv")



Added the latest policy count(policy count of the next month) as the y label for each row

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('output_file.csv')

# Convert year_month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], errors='coerce')

# Sort by agent and date
df.sort_values(['agent_code', 'year_month'], inplace=True)

# Create latest_policy_count as the next month's new_policy_count
df['latest_policy_count'] = df.groupby('agent_code')['new_policy_count'].shift(-1)

# Save to a new CSV
df.to_csv('train_storming_round_with_latest.csv', index=False)

print("✅ New CSV with 'latest_policy_count' created: train_storming_round_with_latest.csv")






# Load the dataset
df = pd.read_csv('train_storming_round_with_latest.csv')

# Remove rows where agent_join_month is 0 or missing
df = df[df['agent_join_month'].astype(str).str.strip() != '0']
df = df[df['agent_join_month'].notna()]

# Convert 'latest_policy_count' to binary: 1 if > 0, else 0
df['latest_policy_count'] = df['latest_policy_count'].apply(lambda x: 1 if x > 0 else 0)

# Save the final processed data
df.to_csv('train_storming_round_binary_target.csv', index=False)

print("Filtered and updated CSV saved as 'train_storming_round_binary_target.csv'")


Got the values for proposals, quotations and customers per each week

In [ ]:
import pandas as pd

# Load your dataset (update the path if needed)
df = pd.read_csv("/Users/rus1ru/Downloads/data-storm-6-0/train_storming_round_binary_target.csv")  # Replace with your actual file path

# Convert cumulative proposal values to weekly values
df["unique_proposals_week_4"] = df["unique_proposals_last_7_days"]
df["unique_proposals_week_3"] = df["unique_proposals_last_15_days"] - df["unique_proposals_last_7_days"]
df["unique_proposals_week_2"] = df["unique_proposals_last_21_days"] - df["unique_proposals_last_15_days"]
df["unique_proposals_week_1"] = df["unique_proposal"] - df["unique_proposals_last_21_days"]

# Convert cumulative quotation values to weekly values
df["unique_quotations_week_4"] = df["unique_quotations_last_7_days"]
df["unique_quotations_week_3"] = df["unique_quotations_last_15_days"] - df["unique_quotations_last_7_days"]
df["unique_quotations_week_2"] = df["unique_quotations_last_21_days"] - df["unique_quotations_last_15_days"]
df["unique_quotations_week_1"] = df["unique_quotations"] - df["unique_quotations_last_21_days"]

# Convert cumulative customer values to weekly values
df["unique_customers_week_4"] = df["unique_customers_last_7_days"]
df["unique_customers_week_3"] = df["unique_customers_last_15_days"] - df["unique_customers_last_7_days"]
df["unique_customers_week_2"] = df["unique_customers_last_21_days"] - df["unique_customers_last_15_days"]
df["unique_customers_week_1"] = df["unique_customers"] - df["unique_customers_last_21_days"]

# Optional: Save the new DataFrame to a file
df.to_csv("output_with_weekly_values.csv", index=False)

print("Weekly values successfully calculated and saved.")

Plotting the Correlation heatmap after adding the lag features and the policy_count for the next month

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data into a pandas DataFrame
data = pd.read_csv('output_with_weekly_values.csv')  # Replace with the actual path to your data file

# Display the first few rows of the dataset
print("First few rows of the dataset:")
print(data.head())

# Basic statistics of the dataset
print("\nBasic statistics of the dataset:")
print(data.describe())

# Check for missing values
print("\nMissing values in the dataset:")
print(data.isnull().sum())



# Correlation heatmap
plt.figure(figsize=(24, 16))

# Select only numeric columns for correlation
numeric_data = data.select_dtypes(include=['float64', 'int64'])
correlation_matrix = numeric_data.corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


Tested all 3 models seperately, as well as the ensemble of all of them to view the performance of each seperately.

In [ ]:
#checking performance of 3 models seperately and as an ensemble

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import classification_report, roc_auc_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier

# ─────────────────────────────────────────────
# 1. Load Data
# ─────────────────────────────────────────────
date_cols = ["year_month", "agent_join_month", "first_policy_sold_month"]
df = pd.read_csv("train_with_weekly_values.csv", parse_dates=date_cols)
df['latest_policy_count'] = (df['latest_policy_count'] > 0).astype(int)

X = df.drop(columns=['latest_policy_count'])
y = df['latest_policy_count']

cat_cols = ["agent_code"]
num_cols = [c for c in X.columns if c not in cat_cols + date_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ─────────────────────────────────────────────
# 2. Transformers
# ─────────────────────────────────────────────
def extract_year_month(X_dates):
    df_dates = pd.DataFrame(X_dates, columns=date_cols)
    feats = []
    for col in date_cols:
        feats.append(df_dates[col].dt.year.values.reshape(-1, 1))
        feats.append(df_dates[col].dt.month.values.reshape(-1, 1))
    return np.hstack(feats)

date_transformer = Pipeline([
    ("extract", FunctionTransformer(extract_year_month, validate=False)),
    ("scale", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num",   StandardScaler(),     num_cols),
    ("cat",   OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("date",  date_transformer,     date_cols)
])

# ─────────────────────────────────────────────
# 3. Model Setup
# ─────────────────────────────────────────────
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale = n_neg / n_pos
print(f"Negative: {n_neg}, Positive: {n_pos}, scale_pos_weight = {scale:.2f}")

xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=scale,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

lgb_model = lgb.LGBMClassifier(
    objective="binary",
    class_weight={0: 1, 1: 1 / scale},
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42
)

cat_model = CatBoostClassifier(
    loss_function="Logloss",
    auto_class_weights='Balanced',
    depth=6,
    learning_rate=0.05,
    iterations=400,
    verbose=0,
    random_seed=42
)

ensemble = StackingClassifier(
    estimators=[
        ("xgb", xgb_model),
        ("lgb", lgb_model),
        ("cat", cat_model)
    ],
    final_estimator=xgb_model,
    passthrough=True,
    n_jobs=1
)

# ─────────────────────────────────────────────
# 4. Evaluation Function
# ─────────────────────────────────────────────
def evaluate_model(name, model):
    print(f"\n{'='*20} Evaluating: {name} {'='*20}")
    pipe = ImbPipeline([
        ("preproc", preprocessor),
        ("smote", SMOTE(sampling_strategy=0.5, random_state=42)),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)

    for dataset, X_, y_ in [("Train", X_train, y_train), ("Test", X_test, y_test)]:
        y_proba = pipe.predict_proba(X_)[:, 1]
        y_pred = pipe.predict(X_)
        print(f"\n-- {dataset} Data --")
        print(classification_report(y_, y_pred, digits=3))
        print(f"{dataset} ROC-AUC: {roc_auc_score(y_, y_proba):.4f}")

# ─────────────────────────────────────────────
# 5. Run Evaluations
# ─────────────────────────────────────────────
evaluate_model("XGBoost", xgb_model)
evaluate_model("LightGBM", lgb_model)
evaluate_model("CatBoost", cat_model)
evaluate_model("Stacked Ensemble", ensemble)


Since in training y=0 is low, changed the models to unbalanced forms(This was the best kaggle submission).

In [ ]:
#BEST CODE


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, roc_auc_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# ─────────────────────────────────────────────
# 1) Load data & create binary target
# ─────────────────────────────────────────────
date_cols = ["year_month", "agent_join_month", "first_policy_sold_month"]

df = pd.read_csv(
    "C:\\Users\\Isitha\\Downloads\\train_with_weekly_values.csv",
    parse_dates=date_cols
)

df['latest_policy_count'] = (df['latest_policy_count'] > 0).astype(int)


# ─────────────────────────────────────────────
# 2) Split into X / y
# ─────────────────────────────────────────────
# keep the raw date columns in X for in-pipeline extraction
drop_cols = ['latest_policy_count']
X = df.drop(columns=drop_cols)
y = df['latest_policy_count']

# ─────────────────────────────────────────────
# 3) Identify column groups
# ─────────────────────────────────────────────
cat_cols = ["agent_code"]
num_cols = [c for c in X.columns
            if c not in cat_cols + date_cols]

# ─────────────────────────────────────────────
# 4) Train/test split
# ────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
# ─────────────────────────────────────────────
# 5) Build date-feature extractor
#    - turns each date column into “year” & “month”
# ─────────────────────────────────────────────
def extract_year_month(X_dates):
    # X_dates: 2D array with columns in the same order as date_cols
    df_dates = pd.DataFrame(X_dates, columns=date_cols)
    feats = []
    for col in date_cols:
        feats.append(df_dates[col].dt.year.values.reshape(-1, 1))
        feats.append(df_dates[col].dt.month.values.reshape(-1, 1))
    return np.hstack(feats)

date_transformer = Pipeline([
    ("extract", FunctionTransformer(extract_year_month, validate=False)),
    ("scale", StandardScaler())
])

# ─────────────────────────────────────────────
# 6) Preprocessing pipelines
# ─────────────────────────────────────────────
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer([
    ("num",   numeric_transformer,    num_cols),
    ("cat",   categorical_transformer,cat_cols),
    ("date",  date_transformer,       date_cols)
], remainder="drop")

# ─────────────────────────────────────────────
# 7) Define base classifiers & stacking ensemble
# ─────────────────────────────────────────────

# 1) After you define X_train, y_train (binary 0/1)
import numpy as np

n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale = n_neg / n_pos

print(f"Negative: {n_neg}, Positive: {n_pos}, scale_pos_weight = {scale:.2f}")


xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=scale,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

lgb_model = lgb.LGBMClassifier(
    objective="binary",
    class_weight={0:1, 1:1/scale},
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42
)

cat_model = CatBoostClassifier(
    loss_function="Logloss",
    auto_class_weights = 'Balanced',
    depth=6,
    learning_rate=0.05,
    iterations=400,
    verbose=0,
    random_seed=42
)

ensemble = StackingClassifier(
    estimators=[
        ("xgb", xgb_model),
        ("lgb", lgb_model),
        ("cat", cat_model)
    ],
    final_estimator=LogisticRegressionCV(),
    passthrough=True,
    n_jobs=1
)

─────────────────────────────────────────
best_results = []


pipe = ImbPipeline([
    ("preproc",  preprocessor),
    ("stack",    ensemble)
])

pipe.fit(X_train, y_train)
y_proba = pipe.predict_proba(X_test)[:,1]
y_preds = pipe.predict(X_test)

print(classification_report(y_test, y_preds, digits=3))
print("Test ROC-AUC:", roc_auc_score(y_test,y_proba))

Oversampling strategy to handle class imbalance

Used SMOTE (Synthetic Minority Over-sampling Technique) to generate synthetic examples of the minority class(ground truth 0) and balance the class distribution in the training data.This helps the model learn the decision boundary more effectively, especially when there is a significant skew between classes (i.e., far more 1s than 0s in target).


SMOTE is applied *after preprocessing* and before training the ensemble model. A fixed sampling_strategy (0.5) is used to control the amount of oversampling.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, roc_auc_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# ─────────────────────────────────────────────
# 1) Load data & create binary target
# ─────────────────────────────────────────────
date_cols = ["year_month", "agent_join_month", "first_policy_sold_month"]

df = pd.read_csv(
    "train_with_weekly_values.csv",
    parse_dates=date_cols
)

df['latest_policy_count'] = (df['latest_policy_count'] > 0).astype(int)


# ─────────────────────────────────────────────
# 2) Split into X / y
# ─────────────────────────────────────────────
# keep the raw date columns in X for in-pipeline extraction
drop_cols = ['latest_policy_count']
X = df.drop(columns=drop_cols)
y = df['latest_policy_count']

# ─────────────────────────────────────────────
# 3) Identify column groups
# ─────────────────────────────────────────────
cat_cols = ["agent_code"]
num_cols = [c for c in X.columns
            if c not in cat_cols + date_cols]

# ─────────────────────────────────────────────
# 4) Train/test split
# ────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
# ─────────────────────────────────────────────
# 5) Build date-feature extractor
#    - turns each date column into “year” & “month”
# ─────────────────────────────────────────────
def extract_year_month(X_dates):
    # X_dates: 2D array with columns in the same order as date_cols
    df_dates = pd.DataFrame(X_dates, columns=date_cols)
    feats = []
    for col in date_cols:
        feats.append(df_dates[col].dt.year.values.reshape(-1, 1))
        feats.append(df_dates[col].dt.month.values.reshape(-1, 1))
    return np.hstack(feats)

date_transformer = Pipeline([
    ("extract", FunctionTransformer(extract_year_month, validate=False)),
    ("scale", StandardScaler())
])

# ─────────────────────────────────────────────
# 6) Preprocessing pipelines
# ─────────────────────────────────────────────
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer([
    ("num",   numeric_transformer,    num_cols),
    ("cat",   categorical_transformer,cat_cols),
    ("date",  date_transformer,       date_cols)
], remainder="drop")

# ─────────────────────────────────────────────
# 7) Define base classifiers & stacking ensemble
# ─────────────────────────────────────────────

# 1) After you define X_train, y_train (binary 0/1)
import numpy as np

n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale = n_neg / n_pos

print(f"Negative: {n_neg}, Positive: {n_pos}, scale_pos_weight = {scale:.2f}")


xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=scale,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

lgb_model = lgb.LGBMClassifier(
    objective="binary",
    class_weight={0:1, 1:1/scale},
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42
)

cat_model = CatBoostClassifier(
    loss_function="Logloss",
    auto_class_weights = 'Balanced',
    depth=6,
    learning_rate=0.05,
    iterations=400,
    verbose=0,
    random_seed=42
)

ensemble = StackingClassifier(
    estimators=[
        ("xgb", xgb_model),
        ("lgb", lgb_model),
        ("cat", cat_model)
    ],
    final_estimator=xgb_model,
    passthrough=True,
    n_jobs=1
)

# ─────────────────────────────────────────────
# 8) SMOTE pipeline & ratio sweep
# ─────────────────────────────────────────────
best_results = []


pipe = ImbPipeline([
    ("preproc",  preprocessor),
    ("smote",   SMOTE(sampling_strategy=0.5, random_state=42)),
    ("stack",    ensemble)
])

pipe.fit(X_train, y_train)
y_proba = pipe.predict_proba(X_test)[:,1]
y_preds = pipe.predict(X_test)

print(classification_report(y_test, y_preds, digits=3))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

We tried SMOTETomek because it combines SMOTE (which balances the classes by oversampling the minority) with Tomek Links (which removes overlapping examples near class boundaries).

This hybrid approach helps not only to address class imbalance, but also to clean noisy or borderline samples, leading to better class separation and potentially improved generalization compared to using SMOTE alone.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import recall_score, precision_score, f1_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.ensemble import StackingClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import BaseEstimator, ClassifierMixin

# ─────────────────────────────────────────────
# 1) Load data & create binary target
# ─────────────────────────────────────────────
date_cols = ["year_month", "agent_join_month", "first_policy_sold_month"]

df = pd.read_csv(
    "train_with_weekly_values.csv",
    parse_dates=date_cols
)

df['latest_policy_count'] = (df['latest_policy_count'] > 0).astype(int)

# ─────────────────────────────────────────────
# 2) Split into X / y
# ─────────────────────────────────────────────
drop_cols = ['latest_policy_count']
X = df.drop(columns=drop_cols)
y = df['latest_policy_count']

# ─────────────────────────────────────────────
# 3) Identify column groups
# ─────────────────────────────────────────────
cat_cols = ["agent_code"]
num_cols = [c for c in X.columns if c not in cat_cols + date_cols]

# ─────────────────────────────────────────────
# 4) Train/test split
# ────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Calculate class imbalance ratio
class_counts = y_train.value_counts()
if len(class_counts) < 2:
    raise ValueError("y_train contains only one class. Check data balance.")
ratio = class_counts[1] / class_counts[0] if class_counts.index[0] == 0 else class_counts[0] / class_counts[1]
print(f"Class ratio (minority/majority): {ratio:.2f}")

# ─────────────────────────────────────────────
# 5) Build date-feature extractor
# ─────────────────────────────────────────────
def extract_year_month(X_dates):
    df_dates = pd.DataFrame(X_dates, columns=date_cols)
    feats = []
    for col in date_cols:
        df_dates[col] = df_dates[col].fillna(pd.Timestamp('2000-01-01'))
        feats.append(df_dates[col].dt.year.values.reshape(-1, 1))
        feats.append(df_dates[col].dt.month.values.reshape(-1, 1))
    return np.hstack(feats)

date_transformer = SklearnPipeline([
    ("extract", FunctionTransformer(extract_year_month, validate=False)),
    ("scale", StandardScaler())
])

# ─────────────────────────────────────────────
# 6) Preprocessing pipeline
# ─────────────────────────────────────────────
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
    ("date", date_transformer, date_cols)
], remainder="drop")

# ─────────────────────────────────────────────
# 7) Define base models and stacking ensemble
# ─────────────────────────────────────────────
xgb = XGBClassifier(random_state=42, scale_pos_weight=ratio)
lgb = LGBMClassifier(random_state=42, class_weight='balanced')
cat = CatBoostClassifier(verbose=0, random_state=42, auto_class_weights='Balanced')

stacking_model = StackingClassifier(
    estimators=[("xgb", xgb), ("lgb", lgb), ("cat", cat)],
    final_estimator=LogisticRegression(),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    passthrough=True
)

# ─────────────────────────────────────────────
# 8) Threshold tuner
# ─────────────────────────────────────────────
class ThresholdTuner(BaseEstimator, ClassifierMixin):
    def __init__(self, base_model, threshold=0.5):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        self.base_model.fit(X, y)
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]  # Probability of class 1
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)


threshold = 0.65
# ─────────────────────────────────────────────
# 9) Full pipeline
# ─────────────────────────────────────────────
pipeline = ImbPipeline([
    ("preproc", preprocessor),
    ("sampler", SMOTETomek(random_state=42)),
    ("model", ThresholdTuner(base_model=stacking_model, threshold=0.65))
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Evaluate
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= threshold).astype(int)

print(classification_report(y_test, y_pred, digits=3))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

Tried running a GridSearch to see which Threshold between 0.35 and 0.7 with gaps of 0.05 would give us the best accuracy. (It didn't finish running tho as it takes a long time to run)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import recall_score, precision_score, f1_score, classification_report, roc_auc_score, make_scorer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.ensemble import StackingClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin

# ─────────────────────────────────────────────
# 1) Load data & create binary target
# ─────────────────────────────────────────────
date_cols = ["year_month", "agent_join_month", "first_policy_sold_month"]

df = pd.read_csv(
    "train_with_weekly_values.csv",
    parse_dates=date_cols
)

df['latest_policy_count'] = (df['latest_policy_count'] > 0).astype(int)

# ─────────────────────────────────────────────
# 2) Split into X / y
# ─────────────────────────────────────────────
drop_cols = ['latest_policy_count']
X = df.drop(columns=drop_cols)
y = df['latest_policy_count']

# ─────────────────────────────────────────────
# 3) Identify column groups
# ─────────────────────────────────────────────
cat_cols = ["agent_code"]
num_cols = [c for c in X.columns if c not in cat_cols + date_cols]

# ─────────────────────────────────────────────
# 4) Train/test split
# ────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Calculate class imbalance ratio
class_counts = y_train.value_counts()
if len(class_counts) < 2:
    raise ValueError("y_train contains only one class. Check data balance.")
ratio = class_counts[1] / class_counts[0] if class_counts.index[0] == 0 else class_counts[0] / class_counts[1]
print(f"Class ratio (minority/majority): {ratio:.2f}")

# ─────────────────────────────────────────────
# 5) Build date-feature extractor
# ─────────────────────────────────────────────
def extract_year_month(X_dates):
    df_dates = pd.DataFrame(X_dates, columns=date_cols)
    feats = []
    for col in date_cols:
        df_dates[col] = df_dates[col].fillna(pd.Timestamp('2000-01-01'))
        feats.append(df_dates[col].dt.year.values.reshape(-1, 1))
        feats.append(df_dates[col].dt.month.values.reshape(-1, 1))
    return np.hstack(feats)

date_transformer = SklearnPipeline([
    ("extract", FunctionTransformer(extract_year_month, validate=False)),
    ("scale", StandardScaler())
])

# ─────────────────────────────────────────────
# 6) Preprocessing pipeline
# ─────────────────────────────────────────────
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
    ("date", date_transformer, date_cols)
], remainder="drop")

# ─────────────────────────────────────────────
# 7) Define base models and stacking ensemble
# ─────────────────────────────────────────────
xgb = XGBClassifier(random_state=42, scale_pos_weight=ratio)
lgb = LGBMClassifier(random_state=42, class_weight='balanced')
cat = CatBoostClassifier(verbose=0, random_state=42, auto_class_weights='Balanced')

stacking_model = StackingClassifier(
    estimators=[("xgb", xgb), ("lgb", lgb), ("cat", cat)],
    final_estimator=LogisticRegression(),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    passthrough=True
)

# ─────────────────────────────────────────────
# 8) Threshold tuner
# ─────────────────────────────────────────────
class ThresholdTuner(BaseEstimator, ClassifierMixin):
    def __init__(self, base_model, threshold=0.5):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        self.base_model.fit(X, y)
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]  # Probability of class 1
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)

# ─────────────────────────────────────────────
# 9) Full pipeline
# ─────────────────────────────────────────────
pipeline = ImbPipeline([
    ("preproc", preprocessor),
    ("sampler", SMOTETomek(random_state=42)),
    ("model", ThresholdTuner(base_model=stacking_model, threshold=0.5))  # Start with default
])

# ─────────────────────────────────────────────
# 10) Tune threshold for class 0 recall
# ─────────────────────────────────────────────
# Define a custom scorer for recall of class 0
def recall_class_0(y_true, y_pred):
    return recall_score(y_true, y_pred, pos_label=0)

recall_scorer = make_scorer(recall_class_0)

# Use GridSearchCV to find the best threshold
param_grid = {'model__threshold': np.arange(0.3, 0.7, 0.05)}  # Test thresholds from 0.3 to 0.65
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    scoring=recall_scorer,  # Optimize for recall of class 0
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

# Print the best threshold
print("Best threshold for class 0 recall:", grid_search.best_params_['model__threshold'])
print("Best recall score for class 0:", grid_search.best_score_)

# Update pipeline with the best threshold
pipeline.set_params(model__threshold=grid_search.best_params_['model__threshold'])

# ─────────────────────────────────────────────
# 11) Evaluate on test set
# ─────────────────────────────────────────────
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= grid_search.best_params_['model__threshold']).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=3))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

Class ratio (minority/majority): 0.18


SVMSMOTE with unbalanaced models gave the best precision, recall and F1 score

In [ ]:
#FINAL
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, roc_auc_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# ─────────────────────────────────────────────
# 1) Load data & create binary target
# ─────────────────────────────────────────────
date_cols = ["year_month", "agent_join_month", "first_policy_sold_month"]

df = pd.read_csv(
    "train_with_weekly_values.csv",
    parse_dates=date_cols
)

df['latest_policy_count'] = (df['latest_policy_count'] > 0).astype(int)


# ─────────────────────────────────────────────
# 2) Split into X / y
# ─────────────────────────────────────────────
# keep the raw date columns in X for in-pipeline extraction
drop_cols = ['latest_policy_count']
X = df.drop(columns=drop_cols)
y = df['latest_policy_count']

# ─────────────────────────────────────────────
# 3) Identify column groups
# ─────────────────────────────────────────────
cat_cols = ["agent_code"]
num_cols = [c for c in X.columns
            if c not in cat_cols + date_cols]

# ─────────────────────────────────────────────
# 4) Train/test split
# ────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
# ─────────────────────────────────────────────
# 5) Build date-feature extractor
#    - turns each date column into “year” & “month”
# ─────────────────────────────────────────────
def extract_year_month(X_dates):
    # X_dates: 2D array with columns in the same order as date_cols
    df_dates = pd.DataFrame(X_dates, columns=date_cols)
    feats = []
    for col in date_cols:
        feats.append(df_dates[col].dt.year.values.reshape(-1, 1))
        feats.append(df_dates[col].dt.month.values.reshape(-1, 1))
    return np.hstack(feats)

date_transformer = Pipeline([
    ("extract", FunctionTransformer(extract_year_month, validate=False)),
    ("scale", StandardScaler())
])

# ─────────────────────────────────────────────
# 6) Preprocessing pipelines
# ─────────────────────────────────────────────
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer([
    ("num",   numeric_transformer,    num_cols),
    ("cat",   categorical_transformer,cat_cols),
    ("date",  date_transformer,       date_cols)
], remainder="drop")

# ─────────────────────────────────────────────
# 7) Define base classifiers & stacking ensemble
# ─────────────────────────────────────────────

# 1) After you define X_train, y_train (binary 0/1)
import numpy as np

n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale = n_neg / n_pos

print(f"Negative: {n_neg}, Positive: {n_pos}, scale_pos_weight = {scale:.2f}")


xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=scale,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

lgb_model = lgb.LGBMClassifier(
    objective="binary",
    class_weight={0:1, 1:1/scale},
    max_depth=6,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    random_state=42
)

cat_model = CatBoostClassifier(
    loss_function="Logloss",
    auto_class_weights = 'Balanced',
    depth=6,
    learning_rate=0.05,
    iterations=400,
    verbose=0,
    random_seed=42
)

ensemble = StackingClassifier(
    estimators=[
        ("xgb", xgb_model),
        ("lgb", lgb_model),
        ("cat", cat_model)
    ],
    final_estimator=LogisticRegressionCV(),
    passthrough=True,
    n_jobs=1
)

# ─────────────────────────────────────────────
# 8) SVMSMOTE pipeline & ratio sweep
# ─────────────────────────────────────────────
best_results = []


pipe = ImbPipeline([
    ("preproc",  preprocessor),
    ('smote', SVMSMOTE(random_state=42)),
    ("stack",    ensemble)
])

pipe.fit(X_train, y_train)
y_proba = pipe.predict_proba(X_test)[:,1]
y_preds = pipe.predict(X_test)

print(classification_report(y_test, y_preds, digits=3))
print("Test ROC-AUC:", roc_auc_score(y_test,y_proba))

Agent Performance Classification
We chose this approach to classify agents based on their average number of new policies sold (new_policy_count). By computing the mean value for each agent_code, we categorized their performance into three levels: Low (less than 14), Medium (14 to 26), and High (greater than 26). This classification provides a clearer understanding of agent effectiveness and is saved to a CSV file for further analysis.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import uuid

# Sample data (from the provided dataset)

df = pd.read_csv("/Users/rus1ru/Downloads/data-storm-6-0/train_with_weekly_values.csv",)


# Alternatively, load from CSV if you have a file
# df = pd.read_csv('your_dataset.csv')

# Aggregate new_policy_count for the agent (using mean)
agent_summary = df.groupby('agent_code')['new_policy_count'].mean().reset_index()
agent_summary.columns = ['agent_code', 'mean_new_policy_count']

# Define performance thresholds
def categorize_performance(count):
    if count < 14:
        return 'Low'
    elif 14 <= count <= 26:
        return 'Medium'
    else:
        return 'High'

# Apply categorization
agent_summary['agent_performance'] = agent_summary['mean_new_policy_count'].apply(categorize_performance)

# Output result
print("Agent Performance Summary:")
print(agent_summary[['agent_code', 'mean_new_policy_count', 'agent_performance']])

# Save result to CSV
agent_summary.to_csv('/Users/rus1ru/Downloads/data-storm-6-0/agent_performance_summary.csv', index=False)
print("\nResult saved to 'agent_performance_summary.csv'")

🔴 Low Performers (below 14 new policies per month)
These agents are either misaligned, undertrained, or disengaged. That stops now.

Immediate One-on-One Coaching: Weekly sessions with team leads focusing on sales fundamentals, mindset, and goal-setting.

Micro Goals: Assign short-term, achievable targets to rewire momentum and build confidence.

Buddy System: Pair with a top-tier agent for on-the-job shadowing. Learning is fastest at the frontline.

Performance Warnings: If no change in 60 days, place under performance improvement plan (PIP). Accountability matters.

////////////////////////////////



🟡 Medium Performers (14–26 policies)
These agents are consistent but not yet great. We push them to break their own ceilings.

Advanced Skill Workshops: Focus on negotiation, objection handling, and upselling. Their raw talent now needs sharpening.

Incentivized Competitions: Introduce tiered weekly contests with exclusive rewards. Medium performers thrive with recognition.

Personalized Feedback: Monthly performance reviews with actionable tweaks, not generic advice.

Promotion Path Visibility: Make clear what numbers they need to hit leadership tracks.


////////////////////////////////


🟢 High Performers (above 26 policies)
These are your 10x agents. Treat them like special forces—but keep raising the bar.

Elite Circle Access: Private roundtables with leadership to co-create strategy. Give them ownership.

Performance Multipliers: Let them lead internal masterclasses. Teaching reinforces greatness.

High-Stakes Challenges: Assign them complex territories or strategic accounts. Trust builds loyalty.

Retention Packages: Customized bonuses, fast-track promotions, and equity/commission escalators.

We designed this tracker to monitor how each agent's performance evolves over time by observing their monthly new_policy_count. Each month, an agent is categorized into one of three performance bands—Low, Medium, or High—based on their output. By comparing the current month’s category with the previous one, we identify whether the agent has improved, declined, remained consistent, or is being tracked for the first time.

This approach shifts the focus from static performance to dynamic growth, giving us insight into how behaviors are changing—not just who performs well right now, but who is on a rising or declining trajectory. It lays the foundation for timely recognition, strategic coaching, and performance accountability at scale.



In [ ]:
import pandas as pd

# Load the dataset with 'agent_code', 'year_month', and 'new_policy_count'
df = pd.read_csv("/Users/rus1ru/Downloads/data-storm-6-0/train_with_weekly_values.csv", parse_dates=["year_month"])

# Step 1: Aggregate by agent and month
monthly_perf = df.groupby(["agent_code", "year_month"])['new_policy_count'].sum().reset_index()

# Step 2: Categorize performance
def categorize(count):
    if count < 14:
        return 'Low'
    elif 14 <= count <= 26:
        return 'Medium'
    else:
        return 'High'

monthly_perf['performance_category'] = monthly_perf['new_policy_count'].apply(categorize)

# Step 3: Sort and compute performance trend
monthly_perf.sort_values(by=["agent_code", "year_month"], inplace=True)
monthly_perf['previous_category'] = monthly_perf.groupby("agent_code")['performance_category'].shift(1)

def detect_change(row):
    if pd.isna(row['previous_category']):
        return "New"
    if row['performance_category'] == row['previous_category']:
        return "No Change"
    elif (row['performance_category'] == "High" and row['previous_category'] in ["Medium", "Low"]) or \
         (row['performance_category'] == "Medium" and row['previous_category'] == "Low"):
        return "Improved"
    else:
        return "Declined"

monthly_perf['status_change'] = monthly_perf.apply(detect_change, axis=1)

# Save or inspect results
monthly_perf.to_csv("/Users/rus1ru/Downloads/data-storm-6-0/agent_progress_tracker.csv", index=False)
print(monthly_perf.head(10))
